**1. Impurity Axioms: Gini & Entropy**

**Symmetry:** Both Gini impurity $G(p) = \sum_{i} p_i (1-p_i) = 1 - \sum p_i^2$ and Entropy $H(p) = - \sum p_i \log_2 p_i$ are symmetric with respect to the permutation of class probabilities. For a binary case with probability $p$ for class 1 and $1-p$ for class 2, $G(p) = 2p(1-p)$ and $H(p) = -p\log p - (1-p)\log(1-p)$. Swapping classes ($p \to 1-p$) yields the exact same value.

**Maximality:** Both functions are strictly concave and reach their maximum when the class distribution is uniform ($p = 0.5$ in binary case). At this point, uncertainty is highest.

**Proof (Gini):** $f(p) = 2p - 2p^2$. Derivative $f'(p) = 2 - 4p = 0 \Rightarrow p=0.5$. Second derivative $f''(p) = -4 < 0$ (Concave/Max).

**Refinement (Splitting reduces impurity):** The strict concavity of these functions ensures that the weighted average impurity of child nodes is always less than or equal to the impurity of the parent node (Jensen’s Inequality). Splitting never increases expected impurity.

**Counterexample:** Consider the Misclassification Error rate, $E(p) = 1 - \max(p_i)$. While intuitive, it is not strictly concave (it is piecewise linear).
- Scenario: Parent node $p=0.8$ (Error=0.2). Split into two pure nodes: Left ($p=0.8$), Right ($p=0.8$). The weighted error remains 0.2. The split provided no "gain" in terms of error rate, even though it might have isolated data structures. Thus, it is often considered a "non-admissible" impurity for growing trees because it lacks the sensitivity of Gini/Entropy.

**2. Bayes-Consistency & Overfitting**

**Bayes-Consistency:** A classification rule is Bayes-consistent if its error rate converges to the Bayes optimal error rate (the theoretical minimum error) as the sample size $n \to \infty$.Greedily grown decision trees are not consistent if they are allowed to grow to full depth (unlimited). A fully grown tree will isolate every single training point (including noise), leading to a variance that does not vanish even with infinite data.However, trees become Bayes-consistent if the tree size is controlled properly as $n$ increases. Specifically, if the leaf size $k_n \to \infty$ such that $k_n / n \to 0$ (e.g., pruning or limiting depth relative to sample size), the local average in each leaf approximates the true posterior probability $P(Y|X)$, converging to the Bayes optimal boundary.

**Toy Distribution:** Consider a dataset generated from two overlapping Gaussian distributions or a "noisy moons" dataset.
- Unlimited Tree: Creates complex, jagged boundaries around every noise point (Overfitting).
- Pruned Tree: Smooths out the boundaries, approximating the true separation between the classes (Approaching Bayes Risk).

**3. Tree Pruning vs. Rule Pruning**

Converting a decision tree to a set of rules (IF-THEN statements) and then pruning the rules (Strategy A) is generally superior to pruning the tree first (Strategy B).

- Context Independence: In a decision tree, a split near the bottom depends entirely on the splits above it. Pruning a node in a tree implicitly removes all its descendants. However, when converted to rules, each path becomes an independent rule. We can prune a condition from a rule (generalize it) without affecting other rules that shared the same parent in the original tree.

- Granularity: Rule pruning allows for a finer level of granularity. We can remove specific antecedents (conditions) from a rule that are redundant or statistically insignificant, effectively "grafting" branches in a way that is topologically difficult to perform directly on the tree structure.

- Readability: The resulting rule set is often much simpler and easier for humans to interpret than the full tree, as it removes the strict hierarchical dependency.

**4. Naive Bayes: Why "Naive"?**

It is called "Naive" because it makes the conditional independence assumption: it assumes that all features $x_i$ are mutually independent given the class label $y$.

$$P(x_1, x_2, ..., x_n | y) \approx \prod_{i=1}^{n} P(x_i | y)$$

In reality, features are almost always correlated (e.g., in text mining, the word "San" is highly correlated with "Francisco"). Ignoring these correlations is a "naive" simplification of reality.

**Major Ideas:**
- **Probabilistic Framework:** It models the posterior probability $P(y|X)$ using Bayes' Theorem.

- **Parameter Efficiency:** Instead of estimating the full joint distribution (which requires exponential data), we only estimate $n$ individual conditional distributions. This reduces variance and makes the model robust to the "curse of dimensionality," especially in small datasets.

**5. Naive Bayes Optimality & MAP Derivation**

**Derivation:**
We want to find the class $\hat{y}$ that maximizes the posterior $P(y|X)$ (Maximum A Posteriori - MAP):
$$\hat{y} = \arg\max_y P(y | x_1, ..., x_n)$$Using Bayes' Theorem:$$P(y | X) = \frac{P(X | y) P(y)}{P(X)}$$Since $P(X)$ is constant for all classes, we maximize the numerator. Applying the "Naive" independence assumption:$$\hat{y} = \arg\max_y P(y) \prod_{i=1}^{n} P(x_i | y)$$
**Dataset Example (Optimality vs. Sub-optimality):**
- **Optimal:** When features are truly independent (e.g., generated from axis-aligned Gaussians). The decision boundary of NB matches the true Bayes boundary.

- **Sub-optimal but Good:** When features are correlated (e.g., rotated Gaussians). NB forces an axis-aligned decision boundary, which is theoretically incorrect (high bias). However, if the correlation affects all classes similarly, the ranking of probabilities might still be correct ($P(y=1|X) > P(y=0|X)$), resulting in high classification accuracy despite poor probability calibration (Rish, 2001).

**6. K-Means: Monotonicity & Local Minima**

**Monotonicity:** Lloyd's algorithm iterates between two steps: 
**(1)** Assignment (assign points to nearest center) and **(2)** Update (recalculate centers). Both steps strictly minimize the objective function (Sum of Squared Errors - SSE). Since SSE is bounded below by 0 and there are a finite number of possible assignments, the algorithm must converge. 

**Local Minima:** Since the algorithm is greedy (coordinate descent), it converges to a local minimum, not necessarily the global one.

**7. DBSCAN: Core, Border, Noise Points**

Definitions:

- Core Point: A point $p$ is a core point if at least minPts points are within distance $\epsilon$ (epsilon) of it (including $p$ itself).

- Border Point: A point that is not a core point but is within distance $\epsilon$ of a core point.

- Noise Point: A point that is neither a core point nor a border point.

**Invariance:** DBSCAN is mostly deterministic. The assignment of Core and Noise points is entirely invariant to the order of data processing. However, the assignment of Border points can depend on order if a border point is reachable from two different clusters (it will be assigned to the first one processed). This is usually a minor effect.

**Merging (Percolation):** As $\epsilon$ increases, the "neighborhoods" of distinct clusters expand. When the $\epsilon$-neighborhoods of two core points from different clusters intersect, a "bridge" of density is formed, and the two clusters merge into one. This is analogous to percolation theory in physics.